# VIX Market Crash — V2 propre
Pipeline autonome : données de clôture → exploration → Random Forest T+1 → XGBoost T+1 → comparaison et backtest.

**Convention** : les variables à la date t sont connues après la publication des clôtures ; les prédictions ne sont pas exécutables à la clôture t. La cible de l'étape 2 reste celle de la V1 : baisse du SPX d'au moins 2 % sur les **5 séances se terminant à t+1**, et **non** baisse sur les 5 prochaines séances. Les seuils VIX sont fixés sur le train uniquement. Le test final reste intact pendant l'entraînement.

## 0. Imports et paramètres

In [ ]:
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth
from google.auth.transport.requests import AuthorizedSession
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

STRATEGY_SHEET_ID = '1XKeBA6hcnPLcdPixX5bWBtbDTBWP-MtK7Av8ead0NN0'
CRASH_DAYS = 5
CRASH_FLAG = -0.02
TEST_FRACTION = 0.20
RANDOM_STATE = 42
VIX_FEATURES = ['Slope_1M_3M', 'Slope_3M_6M', 'Slope_6M_1Y',
                'VIX1M_Change', 'VIX3M_Change', 'VIX6M_Change', 'VIX1Y_Change']

## 1. Chargement Google Sheets et alignement des dates

In [ ]:
def reload_strategy_excel():
    credentials, _ = google.auth.default(
        scopes=['https://www.googleapis.com/auth/drive.readonly'])
    session = AuthorizedSession(credentials)
    response = session.get(
        f'https://www.googleapis.com/drive/v3/files/{STRATEGY_SHEET_ID}/export',
        params={'mimeType': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'},
        timeout=120)
    response.raise_for_status()
    return BytesIO(response.content)

def align_df_dates(raw, names):
    if raw.shape[1] != 2 * len(names):
        raise ValueError(f'Attendu {2 * len(names)} colonnes Date/Valeur, reçu {raw.shape[1]}')
    aligned = []
    for i, name in enumerate(names):
        part = raw.iloc[:, [2*i, 2*i+1]].copy()
        part.columns = ['Date', name]
        part['Date'] = pd.to_datetime(part['Date'], errors='coerce').dt.normalize()
        part[name] = pd.to_numeric(part[name], errors='coerce')
        part = part.dropna(subset=['Date', name]).drop_duplicates('Date', keep='last')
        aligned.append(part.set_index('Date'))
    return pd.concat(aligned, axis=1, join='inner').sort_index()

strategy_excel = reload_strategy_excel()
data_raw = pd.read_excel(strategy_excel, sheet_name='VIX HISTORIQUE',
                         skiprows=3, usecols=[0,1,2,3,5,6,7,8,9,10])
data = align_df_dates(data_raw, ['SPX','VIX1M','VIX3M','VIX6M','VIX1Y'])
assert len(data) > 100 and not data.isna().any().any()
assert not data.index.has_duplicates and data.index.is_monotonic_increasing
print('Observations :', len(data), 'du', data.index.min().date(), 'au', data.index.max().date())
display(data.head())

## 2. Exploration des données

In [ ]:
display(data.describe().T)
print('Valeurs manquantes :\n', data.isna().sum())
print('Corrélations :')
display(data.corr().round(3))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
data['SPX'].plot(ax=axes[0], title='S&P 500')
(data['VIX1M'] - data['VIX3M']).plot(ax=axes[1], title='Pente VIX 1M - 3M')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

## 3. Variables, cibles T+1 et découpage chronologique

In [ ]:
vix_data = data.copy()
vix_data['Slope_1M_3M'] = data['VIX1M'] - data['VIX3M']
vix_data['Slope_3M_6M'] = data['VIX3M'] - data['VIX6M']
vix_data['Slope_6M_1Y'] = data['VIX6M'] - data['VIX1Y']
for name in ['VIX1M','VIX3M','VIX6M','VIX1Y']:
    vix_data[name + '_Change'] = data[name].diff(CRASH_DAYS)
vix_data['SPX_Return_5D'] = data['SPX'].pct_change(CRASH_DAYS)
vix_data = vix_data.dropna(subset=VIX_FEATURES + ['SPX_Return_5D']).copy()

split = int(len(vix_data) * (1 - TEST_FRACTION))
train_dates = vix_data.index[:split]
test_dates = vix_data.index[split:]
thresholds = vix_data.loc[train_dates, VIX_FEATURES].abs().quantile(.75)
vix_data['VIX_Significant_Change'] = vix_data[VIX_FEATURES].abs().gt(thresholds).any(axis=1).astype(int)
vix_data['VIX_Target_T1'] = vix_data['VIX_Significant_Change'].shift(-1)
vix_data['Crash_5D_Historical'] = (vix_data['SPX_Return_5D'] <= CRASH_FLAG).astype(int)
vix_data['Crash_Target_T1'] = vix_data['Crash_5D_Historical'].shift(-1)
print('Train :', train_dates[0].date(), '→', train_dates[-1].date())
print('Test :', test_dates[0].date(), '→', test_dates[-1].date())
display(thresholds.rename('Seuil 75 % (train seulement)').to_frame())

## 4. Random Forest : test chronologique et prédictions OOF

In [ ]:
# Le dernier point du train est exclu : sa cible T+1 est située dans le test.
rf_train = vix_data.iloc[:split-1].dropna(subset=['VIX_Target_T1'])
rf_test = vix_data.iloc[split:].dropna(subset=['VIX_Target_T1'])
stage1_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
stage1_model.fit(rf_train[VIX_FEATURES], rf_train['VIX_Target_T1'].astype(int))
rf_test_proba = stage1_model.predict_proba(rf_test[VIX_FEATURES])[:,1]
print('Random Forest AUC test :', round(roc_auc_score(rf_test['VIX_Target_T1'], rf_test_proba), 3))
print(classification_report(rf_test['VIX_Target_T1'].astype(int), rf_test_proba >= .5, zero_division=0))

# OOF avec 5 plis chronologiques, et gap=1 pour que les labels T+1 du train
# ne contiennent pas d'observations de la fenêtre de validation.
oof = pd.Series(np.nan, index=vix_data.index, dtype=float)
tscv = TimeSeriesSplit(n_splits=5, gap=1)
for idx_train, idx_val in tscv.split(vix_data.iloc[:split]):
    fit_rows = vix_data.iloc[idx_train].dropna(subset=['VIX_Target_T1'])
    val_rows = vix_data.iloc[idx_val]
    if fit_rows['VIX_Target_T1'].nunique() < 2:
        continue
    fold_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    fold_rf.fit(fit_rows[VIX_FEATURES], fit_rows['VIX_Target_T1'].astype(int))
    oof.loc[val_rows.index] = fold_rf.predict_proba(val_rows[VIX_FEATURES])[:,1]

vix_data['RF_Probability_T1'] = oof
vix_data.loc[rf_test.index, 'RF_Probability_T1'] = rf_test_proba
# La dernière observation n'a pas de label RF test, mais sa probabilité peut être calculée.
if vix_data.index[-1] in test_dates:
    vix_data.loc[vix_data.index[-1], 'RF_Probability_T1'] = (
        stage1_model.predict_proba(vix_data.iloc[[-1]][VIX_FEATURES])[0,1])
print('Prédictions OOF pour XGB :', oof.notna().sum())

## 5. XGBoost : avec et sans Random Forest

In [ ]:
features_with_rf = VIX_FEATURES + ['RF_Probability_T1']
# Comparaison équitable : mêmes dates pour les deux modèles.
stage2_train = vix_data.iloc[:split-1].dropna(subset=features_with_rf + ['Crash_Target_T1'])
stage2_test = vix_data.iloc[split:].dropna(subset=features_with_rf + ['Crash_Target_T1'])
y_train = stage2_train['Crash_Target_T1'].astype(int)
y_test = stage2_test['Crash_Target_T1'].astype(int)
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('La cible crash doit comporter les deux classes dans train et test.')
weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=.05,
                         random_state=RANDOM_STATE, scale_pos_weight=weight,
                         eval_metric='logloss', n_jobs=-1)

stage2_model = make_xgb()
stage2_model.fit(stage2_train[features_with_rf], y_train)
p_with = stage2_model.predict_proba(stage2_test[features_with_rf])[:,1]

baseline_model = make_xgb()
baseline_model.fit(stage2_train[VIX_FEATURES], y_train)
p_without = baseline_model.predict_proba(stage2_test[VIX_FEATURES])[:,1]

auc_with = roc_auc_score(y_test, p_with)
auc_without = roc_auc_score(y_test, p_without)
print(f'AUC XGBoost + RF : {auc_with:.3f}')
print(f'AUC XGBoost seul : {auc_without:.3f}')
print(f'Gain AUC : {auc_with-auc_without:+.3f}')
print(classification_report(y_test, p_with >= .5, zero_division=0))
results_v2 = pd.DataFrame({'Actual_Crash': y_test,
                           'Probability_With_RF': p_with,
                           'Probability_Without_RF': p_without},
                          index=stage2_test.index)
display(results_v2.tail())

## 6. Backtest indicatif (sans exécution au close du signal)
Le signal utilise la clôture à t. Pour éviter l'exécution rétroactive, la stratégie ci-dessous prend le rendement **de la clôture t+1 à la clôture t+2** ; une décision d'investissement réelle nécessiterait les prix d'ouverture, les frais, les délais de publication et une convention d'exécution précise. Le seuil 0,5 n'a pas été optimisé sur le test.

In [ ]:
# Rendement exécutable au plus tôt après la séance du signal (hypothèse simplifiée).
future_tradable_return = data['SPX'].shift(-2) / data['SPX'].shift(-1) - 1
bt = results_v2.join(future_tradable_return.rename('Next_Tradable_Return'), how='left')
bt = bt.dropna(subset=['Next_Tradable_Return']).copy()
bt['Invested'] = (bt['Probability_With_RF'] < .5).astype(int)
bt['Strategy_Return'] = bt['Invested'] * bt['Next_Tradable_Return']
bt['SPX_BuyHold'] = (1 + bt['Next_Tradable_Return']).cumprod()
bt['Strategy'] = (1 + bt['Strategy_Return']).cumprod()
bt[['SPX_BuyHold','Strategy']].plot(figsize=(12,5), title='Backtest indicatif hors échantillon (sans frais)')
plt.ylabel('Capital normalisé')
plt.grid(True)
plt.show()
print('Jours investis :', f"{bt['Invested'].mean():.1%}")
print('Performance stratégie :', f"{bt['Strategy'].iloc[-1]-1:.1%}")
print('Performance buy & hold :', f"{bt['SPX_BuyHold'].iloc[-1]-1:.1%}")